# 03. Model and Cost Comparison

**Purpose:** Compare practical trade-offs between models (cost/speed vs accuracy) by running the same prompt on two models.

**Models:**
- Baseline: **gemini-2.5-flash**
- Advanced: **gemini-2.5-pro**

**Key Takeaway:** Model selection is a business decision balancing accuracy, latency, and cost.


## Step 0: Set up environment & authenticate

In [1]:
!git clone https://github.com/alxefremov/esmt-workshop.git

import os, sys, time
from pathlib import Path
PROJECT_ROOT = [
    it for it in [Path("src"), Path('esmt-workshop/src')]
    if (it / 'esmt_workshop').exists()
][0].parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
%pip install -q -r {PROJECT_ROOT}/requirements.txt

import pandas as pd
from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report, publish_to_leaderboard
from esmt_workshop.experiment_logging import load_experiment_history, log_experiment_run
from esmt_workshop.student_api import process_batch_addresses
from esmt_workshop.authenticate import authenticate
from esmt_workshop.preset_params import get_preset_params

STUDENT_EMAIL = "ishika.narang@esmt.berlin"
preset_params = get_preset_params()

fatal: destination path 'esmt-workshop' already exists and is not an empty directory.


## Step 1: Limited-Run Model Comparison (edit **one cell only**)

Use the next cell to compare models quickly on a small subset.

**Do this in order:**
1. Paste your best prompt from Notebook 02 into `STUDENT_PROMPT_TEMPLATE`.
2. **Run the baseline model first**:
   - Set `MODEL_NAME = FLASH_MODEL`
   - Set `STAGE_NAME = 'baseline'`
   - Run the cell → record `micro_accuracy` + `Runtime (sec)`.
3. **Run the advanced model second**:
   - Set `MODEL_NAME = PRO_MODEL`
   - Set `STAGE_NAME = 'advanced'`
   - Run the same cell again → record `micro_accuracy` + `Runtime (sec)`.

**Rule:** Only edit:
- `STUDENT_PROMPT_TEMPLATE` / `PROMPT_TO_RUN`
- `MODEL_NAME`
- `STAGE_NAME`


In [2]:
# -----------------------------
# PROMPT (paste your best prompt from Notebook 02)
# -----------------------------
# Your best prompt (edit this to beat the baseline)
STUDENT_PROMPT_TEMPLATE = """You parse addresses into JSON. Rules: (1) Return ONLY a JSON object matching the schema. (2) All values must be lowercase. (3) Infer country from postal format, language, or state abbreviations if not stated explicitly. (4) Ignore any non-address text in the input.


Schema: {schema}


Examples of correct output format:
- "bastionstraße 6, 47608 geldern" → {{"Town Name": "geldern", "Postal Code": "47608", "Country Code (2 characters)": "de"}}
- "18 school drive, moodus ct 06469" → {{"Town Name": "moodus", "Postal Code": "06469", "Country Code (2 characters)": "us"}}
- "via rancaglia 22, 47899 serravalle" → {{"Town Name": "serravalle", "Postal Code": "47899", "Country Code (2 characters)": "sm"}}


Address: {address}"""


PROMPT_TO_RUN = STUDENT_PROMPT_TEMPLATE


# -----------------------------
# MODEL + STAGE (students change these)
# -----------------------------
FLASH_MODEL = os.getenv('WORKSHOP_BASELINE_MODEL', 'gemini-2.5-flash')
PRO_MODEL = os.getenv('WORKSHOP_ADVANCED_MODEL', 'gemini-2.5-pro')

# Run baseline first:
MODEL_NAME = FLASH_MODEL
STAGE_NAME = 'advanced'   # must be one of: baseline, prompt_tuned, advanced, two_stage

# Then switch to:
# MODEL_NAME = PRO_MODEL
# STAGE_NAME = 'advanced'


# -----------------------------
# FIXED PARAMS (from original notebooks)
# -----------------------------
TEMPERATURE = 0.0
TOP_P = 0.95
TOP_K = 40
USE_GUARDRAILS = False  # default for this notebook

# Execution params (don't change)
MAX_TOKENS = preset_params["MAX_TOKENS"]
MAX_WORKERS = preset_params["MAX_WORKERS"]


# -----------------------------
# DATA (limited run)
# -----------------------------
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')
dev_small = dev_df.head(10).copy()


# -----------------------------
# EXECUTE PIPELINE
# -----------------------------
t0 = time.perf_counter()

pred_df = process_batch_addresses(
    dev_small,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=PROMPT_TO_RUN,
    max_workers=MAX_WORKERS,
)

runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))

# Generate report (same output style as original notebooks)
report = evaluate_predictions(pred_df, dev_small)

print(report['summary'])
display(report['field_metrics'])


Runtime (sec): 366.369
{'rows_ground_truth': 10, 'rows_predictions': 10, 'rows_considered_for_exact_match': 10, 'row_exact_match': 0.0, 'micro_accuracy': 0.0}


,field,matches,total,accuracy
0,Town Name,0,10,0.0
1,Postal Code,0,10,0.0
2,Country Code (2 characters),0,10,0.0


In [3]:
# report['mismatches'] # Also available
# report['usage_metadata'] # Also available
report['merged']

,record_id,Town Name_gt,Postal Code_gt,Country Code (2 characters)_gt,Town Name_pred,Postal Code_pred,Country Code (2 characters)_pred
0,375,Toronto,1234567,US,,,
1,11643,ноокат,723404,kg,,,
2,4521,valle hermoso,5168,ar,,,
3,7987,buchholz in der nordheide,21244,de,,,
4,362,Menlo Park,1234567,GB,,,
5,7167,moodus,06469,us,,,
6,74,Los Angeles,1234567,CA,,,
7,3377,geldern,47608,de,,,
8,7593,miami gardens,33056,us,,,
9,11066,serravalle,47899,sm,,,


## Step 2: Full Dataset Run (your **publishable** result)

Just run and observe you score.

In [ ]:
# -----------------------------
# EXECUTE PIPELINE
# -----------------------------
t0 = time.perf_counter()

pred_df = process_batch_addresses(
    dev_df,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=PROMPT_TO_RUN,
    max_workers=MAX_WORKERS,
)

runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))

report = evaluate_predictions(pred_df, dev_df)

print(report['summary'])
display(report['field_metrics'])


In [ ]:
# report['mismatches'] # Also available
# report['usage_metadata'] # Also available
report['merged']

## Step 3: Publish Placeholder

Later, you will publish the results from **Step 6**. For now, this is a placeholder.

- Save predictions/report artifacts
- Submit score via the workshop publishing mechanism (to be provided)


In [ ]:
publish_to_leaderboard(report, STUDENT_EMAIL)